In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CÉLULA 1 — INSTALAÇÃO DE DEPENDÊNCIAS              ║
# ║  Execute apenas UMA VEZ por sessão do Colab.        ║
# ╚══════════════════════════════════════════════════════╝

print('Instalando pacotes Python...')
!pip install PyPDF2 pytesseract pdf2image Pillow requests -q

print('Instalando Whisper (OpenAI)...')
!pip install git+https://github.com/openai/whisper.git -q

print('Instalando ffmpeg, Tesseract e Poppler...')
!sudo apt-get update -qq
!sudo apt-get install -y ffmpeg tesseract-ocr tesseract-ocr-por poppler-utils -qq

print('\n✅ Todas as dependências instaladas com sucesso.')

Instalando pacotes Python...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.2 MB/s eta 0:00:00
Instalando Whisper (OpenAI)...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Instalando ffmpeg, Tesseract e Poppler...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 5.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
(Reading database ... 122403 files and direct

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CÉLULA 2 — MONTAGEM DO DRIVE E IMPORTS             ║
# ╚══════════════════════════════════════════════════════╝

from google.colab import drive
drive.mount('/content/drive')

import os
import PyPDF2
import subprocess
import shutil
import time
from datetime import datetime
from pathlib import Path

print('✅ Drive montado e bibliotecas importadas.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive montado e bibliotecas importadas.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CÉLULA 3 — CONFIGURAÇÃO DO LOTE                                    ║
# ║                                                                      ║
# ║  Edite apenas esta célula para configurar suas aulas.               ║
# ║                                                                      ║
# ║  CAMPOS DE CADA AULA:                                                ║
# ║  • nome_saida      → "03 - Tema Da Aula" (sem extensão)            ║
# ║  • caminhos_audios → Lista de áudios (1 parte ou várias partes)     ║
# ║  • caminho_pdf     → Caminho do PDF de slides (ou None se não tiver)║
# ║  • prompt_whisper  → Termos médicos para guiar o Whisper            ║
# ╚══════════════════════════════════════════════════════════════════════╝

import re
import os
import subprocess
import sys
from pathlib import Path

# ──────────────────────────────────────────────
# PASTA DE SAÍDA NO DRIVE (onde os .txt vão)
# ──────────────────────────────────────────────
PASTA_SAIDA_DRIVE = "/content/drive/MyDrive/Logística - Drive/Transcrições/Transcricoes_Medicina"

# ──────────────────────────────────────────────
# OPÇÕES AVANÇADAS DO WHISPER
# ──────────────────────────────────────────────
WHISPER_CONFIG = {
    "model":                      "large-v3",
    "language":                   "Portuguese",
    "temperature":                "0",
    "condition_on_previous_text": "False",
}


# ──────────────────────────────────────────────
# LOTE DE AULAS
# ──────────────────────────────────────────────
PREFIXO_AUDIO = "/content/drive/MyDrive/Áudios aulas/"

aulas_para_processar = [

    # ── AULA 1 ──
    {
        "nome_saida": "LHM - Doenças Relacionadas Ao Trabalho",
        "caminhos_audios": [
            PREFIXO_AUDIO + "Conferência - Doenças relacionadas ao trabalho.m4a",
        ],
        "caminho_pdf": "/content/drive/MyDrive/2026.2 - M6/Doenças e Meio Ambiente/P5 - Saúde do Trabalhador/NOÇÕES BÁSICAS DE MEDICINA DO TRABALHO.pptx.pdf",
        "prompt_whisper": "Aula de Medicina: Medicina do trabalho, pneumoconiose, silicose, asbestose, mesotelioma, saturnismo, hidrargirismo, benzenismo, asma ocupacional, PAIR, LER, DORT, síndrome de burnout, nexo causal, CAT, ergonomia, toxicologia ocupacional, bissinose, beriliose, pneumonite de hipersensibilidade, radiação ionizante, leucopenia, anemia aplástica, chumbo, mercúrio, cromo, dermatite de contato ocupacional, surdez neurossensorial, neuropatia periférica, epicondilite, tenossinovite de De Quervain, síndrome do túnel do carpo, vibração de corpo inteiro, ruído ocupacional, poeiras inorgânicas, espirometria, audiometria, exame toxicológico, limite de tolerância, insalubridade",
    },

    # ── AULA 2 ──
    {
        "nome_saida": "LHM - Drogas De Abuso",
        "caminhos_audios": [
            PREFIXO_AUDIO + "Conferência - drogas de abuso - parte 01 20260527-141035.m4a",
            PREFIXO_AUDIO + "Conferência - drogas de abuso - parte 02 20260527-141653.m4a",
        ],
        "caminho_pdf": "/content/drive/MyDrive/2026.2 - M6/Doenças e Meio Ambiente/P2 - Drogas Ilícitas/AULA - Drogas Ilícitas.pdf",
        "prompt_whisper": "Aula de Medicina: Toxicologia, psiquiatria, dependência química, tolerância, síndrome de abstinência, intoxicação aguda, overdose, cocaína, crack, maconha, THC, canabidiol, anfetaminas, metanfetamina, MDMA, ecstasy, opioides, heroína, fentanil, morfina, naloxona, flumazenil, benzodiazepínicos, barbitúricos, LSD, alucinógenos, cetamina, álcool, delirium tremens, síndrome de Wernicke-Korsakoff, hepatopatia alcoólica, miose, midríase, depressão respiratória, taquicardia, hipertensão, agitação psicomotora, psicose tóxica, dopamina, sistema de recompensa",
    },
]

# ──────────────────────────────────────────────
# Instalação de dependências do sistema (OCR)
# ──────────────────────────────────────────────
subprocess.run(
    ['sudo', 'apt-get', 'install', '-y', '-qq',
     'poppler-utils', 'tesseract-ocr', 'tesseract-ocr-por'],
    check=True
)
print('✅ Poppler e Tesseract (português) instalados.')

def _instalar_se_ausente(pacote_pip, import_name=None):
    import importlib
    nome = import_name or pacote_pip
    try:
        importlib.import_module(nome)
    except ImportError:
        print(f'[DEP] Instalando {pacote_pip}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', pacote_pip, '-q'], check=True)

_instalar_se_ausente('pytesseract')
_instalar_se_ausente('pdf2image')
_instalar_se_ausente('Pillow', 'PIL')

# ──────────────────────────────────────────────
# Função que formata o nome final do arquivo
# ──────────────────────────────────────────────
def formatar_nome_saida(nome_saida: str) -> str:
    return f'{nome_saida.strip()} (Resumo)'

# ──────────────────────────────────────────────
# Validação de caminhos
# ──────────────────────────────────────────────
print('\nValidando configuração do lote...\n')
erros_encontrados = False

for i, aula in enumerate(aulas_para_processar):
    nome_final = formatar_nome_saida(aula["nome_saida"])
    print(f'[Aula {i+1}] {nome_final}')

    for j, caminho_audio in enumerate(aula["caminhos_audios"]):
        existe = os.path.exists(caminho_audio)
        status = '✅' if existe else '❌ ARQUIVO NÃO ENCONTRADO'
        print(f'  Áudio {j+1}: {status} → {caminho_audio}')
        if not existe:
            erros_encontrados = True

    if aula["caminho_pdf"]:
        existe_pdf = os.path.exists(aula["caminho_pdf"])
        status_pdf = '✅' if existe_pdf else '❌ ARQUIVO NÃO ENCONTRADO'
        print(f'  PDF:     {status_pdf} → {aula["caminho_pdf"]}')
        if not existe_pdf:
            erros_encontrados = True
    else:
        print('  PDF:     ⚠️  Nenhum PDF configurado para esta aula.')

    priming = aula.get("prompt_whisper", "").strip()
    if priming:
        print(f'  Priming: ✅ ({len(priming.split(","))} termos)')
    else:
        print('  Priming: ⚠️  Não definido — Whisper rodará sem priming.')

    print()

if erros_encontrados:
    print('⛔ ATENÇÃO: Corrija os caminhos acima antes de executar a Célula 4.')
else:
    print(f'✅ Tudo certo! {len(aulas_para_processar)} aula(s) configurada(s). Execute a Célula 4.')

✅ Poppler e Tesseract (português) instalados.

Validando configuração do lote...

[Aula 1] LHM - Doenças Relacionadas Ao Trabalho (Resumo)
  Áudio 1: ✅ → /content/drive/MyDrive/Logística - Drive/Transcrições/Áudios aulas/Conferência - Doenças relacionadas ao trabalho.m4a
  PDF:     ✅ → /content/drive/MyDrive/2026.2 - M6/Doenças e Meio Ambiente/P5 - Saúde do Trabalhador/NOÇÕES BÁSICAS DE MEDICINA DO TRABALHO.pptx.pdf
  Priming: ✅ (40 termos)

[Aula 2] LHM - Drogas De Abuso (Resumo)
  Áudio 1: ✅ → /content/drive/MyDrive/Logística - Drive/Transcrições/Áudios aulas/Conferência - drogas de abuso - parte 01 20260527-141035.m4a
  Áudio 2: ✅ → /content/drive/MyDrive/Logística - Drive/Transcrições/Áudios aulas/Conferência - drogas de abuso - parte 02 20260527-141653.m4a
  PDF:     ✅ → /content/drive/MyDrive/2026.2 - M6/Doenças e Meio Ambiente/P2 - Drogas Ilícitas/AULA - Drogas Ilícitas.pdf
  Priming: ✅ (40 termos)

✅ Tudo certo! 2 aula(s) configurada(s). Execute a Célula 4.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CÉLULA 4 — MOTOR DE EXECUÇÃO                                       ║
# ║  Não edite esta célula. Apenas execute-a.                           ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ── Imports principais ────────────────────────────────────────────────
import os
import re
import shutil
import time
import subprocess
import PyPDF2
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
from datetime import datetime
from pathlib import Path

# ── Constantes internas ───────────────────────────────────────────────
DIR_TEMP            = "/content/_whisper_temp"
SEPARADOR           = "\n\n[--- PAUSA NA GRAVAÇÃO / CONTINUAÇÃO DA AULA ---]\n\n"
LIMITE_CHARS_PAGINA = 50


# ═════════════════════════════════════════════════════════════════════
# FUNÇÕES AUXILIARES
# ═════════════════════════════════════════════════════════════════════

def log(msg: str, nivel: str = 'INFO'):
    ts = datetime.now().strftime('%H:%M:%S')
    prefixos = {'INFO': '   ', 'OK': '✅ ', 'WARN': '⚠️ ', 'ERR': '❌ ', 'STEP': '▶️ ', 'OCR': '🔍 '}
    print(f"[{ts}] {prefixos.get(nivel, '')} {msg}")


def obter_duracao_audio_segundos(caminho_audio: str) -> float:
    """
    Retorna a duração em segundos usando ffprobe.
    Retorna 0.0 em caso de falha — não interrompe o pipeline.
    """
    try:
        resultado = subprocess.run(
            [
                'ffprobe', '-v', 'error',
                '-show_entries', 'format=duration',
                '-of', 'default=noprint_wrappers=1:nokey=1',
                caminho_audio
            ],
            capture_output=True, text=True
        )
        return float(resultado.stdout.strip())
    except Exception as e:
        log(f'Não foi possível obter duração de {Path(caminho_audio).name}: {e}', 'WARN')
        return 0.0


def transcrever_audio(caminho_audio, parte_num, total_partes, prompt, config):
    log(f'Transcrevendo parte {parte_num}/{total_partes}: {Path(caminho_audio).name}', 'STEP')
    if not os.path.exists(caminho_audio):
        raise FileNotFoundError(f'Áudio não encontrado: {caminho_audio}')
    os.makedirs(DIR_TEMP, exist_ok=True)

    comando = [
        'whisper', caminho_audio,
        '--model',                      config['model'],
        '--language',                   config['language'],
        '--temperature',                config['temperature'],
        '--condition_on_previous_text', config['condition_on_previous_text'],
        '--device',                     'cuda',
        '--output_dir',                 DIR_TEMP,
        '--output_format',              'txt',
    ]
    if prompt and prompt.strip():
        comando += ['--initial_prompt', prompt]

    resultado = subprocess.run(comando, capture_output=True, text=True)
    if resultado.returncode != 0:
        raise RuntimeError(f'Whisper falhou (código {resultado.returncode}):\n{resultado.stderr[-1000:]}')

    nome_base   = Path(caminho_audio).stem
    caminho_txt = os.path.join(DIR_TEMP, nome_base + '.txt')
    if not os.path.exists(caminho_txt):
        raise FileNotFoundError(
            f'O Whisper não gerou o arquivo esperado: {caminho_txt}\n'
            f'Arquivos encontrados em {DIR_TEMP}: {os.listdir(DIR_TEMP)}'
        )
    with open(caminho_txt, 'r', encoding='utf-8') as f:
        texto = f.read()
    os.remove(caminho_txt)
    log(f'Parte {parte_num} transcrita. Caracteres: {len(texto):,}', 'OK')
    return texto


def ocr_pagina(imagem: Image.Image, num_pagina: int) -> str:
    """Aplica OCR em uma imagem de página usando Tesseract."""
    try:
        texto = pytesseract.image_to_string(imagem, lang='por')
        return texto.strip()
    except Exception as e:
        log(f'OCR falhou na página {num_pagina}: {e}', 'WARN')
        return ''


def extrair_texto_pdf(caminho_pdf: str) -> str:
    """
    Extrai texto de um PDF com detecção automática de páginas escaneadas.
    Estratégia: PyPDF2 primeiro; OCR com Tesseract nas páginas insuficientes.
    """
    if not caminho_pdf:
        return 'Nenhum slide fornecido para esta aula.'
    if not os.path.exists(caminho_pdf):
        return f'AVISO: PDF não encontrado no caminho especificado: {caminho_pdf}'

    try:
        with open(caminho_pdf, 'rb') as f:
            leitor            = PyPDF2.PdfReader(f)
            total_paginas     = len(leitor.pages)
            textos_por_pagina = []
            for pagina in leitor.pages:
                t = pagina.extract_text()
                textos_por_pagina.append(t.strip() if t else '')

        log(f'PDF aberto. Total de páginas: {total_paginas}')

        paginas_para_ocr = [
            i for i, t in enumerate(textos_por_pagina)
            if len(t) < LIMITE_CHARS_PAGINA
        ]
        digitais = total_paginas - len(paginas_para_ocr)
        log(f'Páginas digitais: {digitais} | Páginas para OCR: {len(paginas_para_ocr)}')

        if paginas_para_ocr:
            log(f'Iniciando OCR em {len(paginas_para_ocr)} página(s)...', 'OCR')
            imagens = convert_from_path(
                caminho_pdf,
                dpi=200,
                first_page=min(paginas_para_ocr) + 1,
                last_page=max(paginas_para_ocr) + 1
            )
            indice_imagem = {
                num_pag: imagens[i]
                for i, num_pag in enumerate(
                    range(min(paginas_para_ocr), max(paginas_para_ocr) + 1)
                )
                if num_pag in paginas_para_ocr
            }
            ocr_ok = ocr_falha = 0
            for num_pag in paginas_para_ocr:
                imagem = indice_imagem.get(num_pag)
                if imagem:
                    texto_ocr = ocr_pagina(imagem, num_pag + 1)
                    if texto_ocr:
                        textos_por_pagina[num_pag] = texto_ocr
                        ocr_ok += 1
                    else:
                        textos_por_pagina[num_pag] = f'[PÁGINA {num_pag + 1}: OCR não extraiu conteúdo legível]'
                        ocr_falha += 1
            log(f'OCR concluído. Sucesso: {ocr_ok} | Falha: {ocr_falha}', 'OCR')

        texto_total = ''
        for num_pagina, texto in enumerate(textos_por_pagina, start=1):
            if texto:
                texto_total += f'--- Slide {num_pagina} ---\n{texto}\n\n'

        if not texto_total.strip():
            return (
                'AVISO: Nenhum conteúdo foi extraído do PDF — '
                'nem via leitura direta nem via OCR. '
                'O resumo será gerado apenas com base na transcrição da aula.'
            )

        log(f'Extração concluída. Total de caracteres: {len(texto_total):,}', 'OK')
        return texto_total

    except Exception as e:
        log(f'Erro inesperado na extração do PDF: {type(e).__name__}: {e}', 'ERR')
        return (
            f'AVISO: Falha na extração do PDF ({type(e).__name__}: {e}). '
            'O resumo será gerado apenas com base na transcrição da aula.'
        )


def montar_payload(transcricao, slides, nomes_audios, duracao_segundos_total):
    cabecalho_audios = ",".join(nomes_audios)
    h = int(duracao_segundos_total // 3600)
    m = int((duracao_segundos_total % 3600) // 60)
    s = int(duracao_segundos_total % 60)
    if duracao_segundos_total == 0.0:
        duracao_str = "Não foi possível determinar (ffprobe falhou)"
    elif h > 0:
        duracao_str = f"{h}h {m}m {s}s"
    elif m > 0:
        duracao_str = f"{m}m {s}s"
    else:
        duracao_str = f"{s}s"
    return (
        f'**AUDIOS_ORIGEM:**{cabecalho_audios}\n\n'
        f'**DURACAO_TOTAL_DA_AULA:**{duracao_str}\n\n'
        '**TRANSCRIÇÃO_DA_AULA_EM_TEXTO_BRUTO:**\n'
        f'{transcricao}\n\n'
        '**CONTEÚDO_DOS_SLIDES_EM_TEXTO:**\n'
        f'{slides}'
    )


def salvar_txt_drive(conteudo, nome_saida, pasta_destino):
    os.makedirs(pasta_destino, exist_ok=True)
    caminho_final = os.path.join(pasta_destino, f'{nome_saida}.txt')
    with open(caminho_final, 'w', encoding='utf-8') as f:
        f.write(conteudo)
    tamanho_kb = os.path.getsize(caminho_final) / 1024
    log(f'Arquivo salvo: {caminho_final} ({tamanho_kb:.1f} KB)', 'OK')
    return caminho_final


# ═════════════════════════════════════════════════════════════════════
# EXECUÇÃO PRINCIPAL
# ═════════════════════════════════════════════════════════════════════

tempo_inicio_lote = time.time()
relatorio_final   = []
separador_visual  = '=' * 65

print(separador_visual)
print('  PIPELINE DE TRANSCRIÇÃO MÉDICA — v2.4 (CUDA + OCR + Duração + Priming Auto)')
print(f'  Início: {datetime.now().strftime("%d/%m/%Y %H:%M:%S")}')
print(f'  Aulas no lote: {len(aulas_para_processar)}')
print(separador_visual)

for idx_aula, aula in enumerate(aulas_para_processar, start=1):
    nome   = formatar_nome_saida(aula['nome_saida'])
    audios = aula['caminhos_audios']
    pdf    = aula.get('caminho_pdf')
    prompt = aula.get('prompt_whisper', '').strip()

    print(f'\n{separador_visual}')
    print(f'  AULA {idx_aula}/{len(aulas_para_processar)}: {nome}')
    print(separador_visual)

    tempo_inicio_aula      = time.time()
    status_aula            = 'SUCESSO'
    motivo_falha           = ''
    duracao_total_segundos = 0.0
    texto_slides           = ''

    try:
        # ── PASSO 1: Transcrição ──────────────────────────────────────
        # Executada primeiro para garantir que a GPU (VRAM) seja usada\n        # antes de qualquer outra biblioteca carregar na RAM do sistema.\n
        log(f'PASSO 1/3 — Transcrição de áudio ({len(audios)} parte(s))', 'STEP')
        transcricao_completa = ''

        for i, caminho_audio in enumerate(audios, start=1):
            duracao_parte = obter_duracao_audio_segundos(caminho_audio)
            duracao_total_segundos += duracao_parte
            if duracao_parte > 0:
                log(f'Duração da parte {i}: {int(duracao_parte // 60)}m {int(duracao_parte % 60)}s')

            texto_parte = transcrever_audio(
                caminho_audio=caminho_audio,
                parte_num=i,
                total_partes=len(audios),
                prompt=prompt,
                config=WHISPER_CONFIG
            )
            transcricao_completa += texto_parte
            if i < len(audios):
                transcricao_completa += SEPARADOR

        log(f'Transcrição total: {len(transcricao_completa):,} caracteres.', 'OK')

        if duracao_total_segundos > 0:
            h = int(duracao_total_segundos // 3600)
            m = int((duracao_total_segundos % 3600) // 60)
            s = int(duracao_total_segundos % 60)
            duracao_log = f'{h}h {m}m {s}s' if h > 0 else f'{m}m {s}s'
            log(f'Duração total da aula ({len(audios)} parte(s)): {duracao_log}', 'OK')
        else:
            log('Duração não determinada (ffprobe indisponível ou falhou).', 'WARN')

        # ── PASSO 2: Extração do PDF ──────────────────────────────────
        # Executada após a transcrição — Tesseract e pdf2image sobem\n        # somente depois que o Whisper já liberou a VRAM.\n
        log('PASSO 2/3 — Extração de texto do PDF (com detecção OCR)', 'STEP')
        texto_slides = extrair_texto_pdf(pdf)
        # ── PASSO 3: Montagem e salvamento ───────────────────────────
        log('PASSO 3/3 — Montagem do payload e envio ao Drive', 'STEP')
        nomes_audios    = [Path(c).name for c in audios]
        documento_final = montar_payload(
            transcricao_completa,
            texto_slides,
            nomes_audios,
            duracao_total_segundos
        )
        caminho_salvo = salvar_txt_drive(documento_final, nome, PASTA_SAIDA_DRIVE)

    except FileNotFoundError as e:
        log(f'Arquivo não encontrado: {e}', 'ERR')
        status_aula  = 'FALHA'
        motivo_falha = f'FileNotFoundError: {e}'

    except RuntimeError as e:
        log(f'Erro no Whisper: {e}', 'ERR')
        status_aula  = 'FALHA'
        motivo_falha = f'RuntimeError: {str(e)[:200]}'

    except Exception as e:
        log(f'Erro inesperado: {type(e).__name__}: {e}', 'ERR')
        status_aula  = 'FALHA'
        motivo_falha = f'{type(e).__name__}: {str(e)[:200]}'

    finally:
        if os.path.exists(DIR_TEMP):
            shutil.rmtree(DIR_TEMP)

    duracao_aula = time.time() - tempo_inicio_aula
    relatorio_final.append({
        'nome':    nome,
        'status':  status_aula,
        'duracao': duracao_aula,
        'motivo':  motivo_falha,
    })

# ── RELATÓRIO FINAL ───────────────────────────────────────────────────
duracao_lote = time.time() - tempo_inicio_lote

print(f'\n{separador_visual}')
print('  RELATÓRIO FINAL DO LOTE')
print(separador_visual)

sucesso = sum(1 for r in relatorio_final if r['status'] == 'SUCESSO')
falhas  = sum(1 for r in relatorio_final if r['status'] == 'FALHA')

for r in relatorio_final:
    icone = '✅' if r['status'] == 'SUCESSO' else '❌'
    mins  = int(r['duracao'] // 60)
    segs  = int(r['duracao'] % 60)
    print(f"  {icone} {r['nome']} ({mins}m {segs}s)")
    if r['motivo']:
        print(f"     └─ Motivo: {r['motivo']}")

print()
print(f'  Resultado: {sucesso} sucesso(s), {falhas} falha(s)')
print(f'  Duração total do lote: {int(duracao_lote // 60)}m {int(duracao_lote % 60)}s')
print(f'  Arquivos prontos em: {PASTA_SAIDA_DRIVE}')
print(separador_visual)

if sucesso > 0:
    print(f'\n✅ {sucesso} arquivo(s) depositado(s) na fila do Google Drive.')
    print('   O Apps Script irá processá-los automaticamente no próximo ciclo agendado.')
if falhas > 0:
    print(f'\n⚠️  {falhas} aula(s) falharam. Verifique os erros acima e re-execute somente as aulas com falha.')

  PIPELINE DE TRANSCRIÇÃO MÉDICA — v2.4 (CUDA + OCR + Duração + Priming Auto)
  Início: 10/06/2026 17:39:48
  Aulas no lote: 2

  AULA 1/2: LHM - Doenças Relacionadas Ao Trabalho (Resumo)
[17:39:48] ▶️  PASSO 1/3 — Transcrição de áudio (1 parte(s))
[17:39:50]     Duração da parte 1: 73m 51s
[17:39:50] ▶️  Transcrevendo parte 1/1: Conferência - Doenças relacionadas ao trabalho.m4a
[18:08:26] ✅  Parte 1 transcrita. Caracteres: 56,257
[18:08:26] ✅  Transcrição total: 56,257 caracteres.
[18:08:26] ✅  Duração total da aula (1 parte(s)): 1h 13m 51s
[18:08:26] ▶️  PASSO 2/3 — Extração de texto do PDF (com detecção OCR)


[18:08:28]     PDF aberto. Total de páginas: 45
[18:08:28]     Páginas digitais: 32 | Páginas para OCR: 13
[18:08:28] 🔍  Iniciando OCR em 13 página(s)...
[18:08:42] 🔍  OCR concluído. Sucesso: 12 | Falha: 1
[18:08:42] ✅  Extração concluída. Total de caracteres: 12,955
[18:08:42] ✅  Priming manual utilizado: 40 termos.
[18:08:42] ▶️  PASSO 3/3 — Montagem do payload e envio ao Drive
[18:08:42] ✅  Arquivo salvo: /content/drive/MyDrive/Logística - Drive/Transcrições/Transcricoes_Medicina/LHM - Doenças Relacionadas Ao Trabalho (Resumo).txt (70.9 KB)

  AULA 2/2: LHM - Drogas De Abuso (Resumo)
[18:08:42] ▶️  PASSO 1/3 — Transcrição de áudio (2 parte(s))
[18:08:44]     Duração da parte 1: 87m 21s
[18:08:44] ▶️  Transcrevendo parte 1/2: Conferência - drogas de abuso - parte 01 20260527-141035.m4a
[18:40:12] ✅  Parte 1 transcrita. Caracteres: 60,751
[18:40:14]     Duração da parte 2: 12m 14s
[18:40:14] ▶️  Transcrevendo parte 2/2: Conferência - drogas de abuso - parte 02 20260527-141653.m4a
[

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  CÉLULA 5 — ACIONAMENTO DO WEBHOOK                  ║
# ║  Aciona o Apps Script para gerar resumos e cards    ║
# ╚══════════════════════════════════════════════════════╝

import requests

WEBHOOK_URL = "https://script.google.com/macros/s/AKfycbwNaI5m7rKM-i35fQMt5jsSbdinSkpZFYNS3_g-8RSPamfFvp-6UJz-Xwso-7nEObbtmQ/exec"

print('📡 Acionando processamento de resumos no Google Apps Script...')
try:
    response = requests.post(WEBHOOK_URL, allow_redirects=True)
    if response.status_code == 200:
        resultado = response.json()
        if resultado.get('status') == 'sucesso':
            print('✅ Sucesso! O Apps Script concluiu a geração do resumo e dos flashcards.')
        else:
            print(f'⚠️ O Apps Script retornou um aviso: {resultado.get("mensagem")}')
    else:
        print(f'❌ Falha de comunicação. Código HTTP: {response.status_code}')
        print(response.text)
except Exception as e:
    print(f'❌ Falha ao conectar ao Webhook: {e}')